## 1. Setup

In [1]:
from pathlib import Path
import sys
import json
import gzip
import re
import ast
import hashlib
import random
import inspect
import time
from collections import Counter
from traceback import format_exception_only
from typing import Any, Mapping

import pandas as pd
from tqdm.auto import tqdm
# https://www.kaggle.com/datasets/senju14/ocr-dataset-of-multi-type-documents/data


## 2. Configuration

In [2]:
# Repository root. Run the notebook from the project root if possible.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and Path("/mnt/data/project").exists():
    PROJECT_ROOT = Path("/mnt/data/project")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Required user-provided paths.
INPUT_DIR = PROJECT_ROOT / "data" / "raw" / "multi_docs"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "processed" / "multi_docs"

DATASET_NAME = "invoices"

# Split assignment. Set SPLIT_SEED=None to put every record into split="all" only.
SPLIT_SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10
DEFAULT_SPLIT = "all"

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

# Raw entity key → canonical label.
ENTITY_LABEL_MAP = {
    "company": "Company",
    "date": "Date",
    "address": "Address",
    "total": "Total",
}

# Conversion controls.
JOIN_BOXES_WITH = "\n"
PAGE_ID = 0
OCR_UNIT = "box_text"
KEEP_UNMATCHED_IN_CANONICAL_LABELS = False  # recommended False for training/evaluation readiness

# Alignment controls.
MIN_TOKEN_OVERLAP = 0.5
DROP_UNAVAILABLE = True
DROP_LOOSE_MISMATCH = False

WRITE_SERIALIZED = True
INCLUDE_HEAVY_SERIALIZERS = False
HEAVY_SERIALIZERS = { }

# Diagnostics/validation.
VALIDATE_FIRST_N_DOCS_PER_DATASET = 10
ASSESS_QUALITY_FIRST_N_DOCS = 25

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print("input exists:", INPUT_DIR.exists())

assert (PROJECT_ROOT / "src").exists(), "Run this notebook from the repository root containing src/."
assert INPUT_DIR.exists(), f"Input directory does not exist: {INPUT_DIR}"

PROJECT_ROOT: /home/vios/PycharmProjects/serialization-strategies
INPUT_DIR: /home/vios/PycharmProjects/serialization-strategies/data/raw/multi_docs
OUTPUT_ROOT: /home/vios/PycharmProjects/serialization-strategies/data/processed/multi_docs
src exists: True
input exists: True


## 3. Project imports

In [3]:
from src.preprocessing.schema import (
    Annotation,
    CanonicalDocument,
    OCRBlock,
    OCRToken,
)

from src.preprocessing.quality import (
    assess_document_quality,
    filter_annotations_for_training,
)

from src.preprocessing.alignment import (
    add_token_labels,
    bio_to_spans,
)

from src.serialization import (
    IGNORE_LABEL,
    PlainTextSerializer,
    PageAwareSerializer,
    BlockAwareSerializer,
    LineAwareSerializer,
    RowColBucketSerializer,
    BBoxTokenSerializer,
    ColumnAwareSerializer,
    XYCutAwareSerializer,
    LMDXCoordSuffixSerializer,
    CompactBBoxTokenSerializer,
    PrecedenceGraphOrderSerializer,
    KeyValueRowPairsSerializer,
    KeyValueAnchorPairsSerializer,
    collapse_serialized_labels_to_original,
)

print("Project imports OK")

Project imports OK


## 4. File loading

In [6]:
def is_json_path(path: Path) -> bool:
    name = path.name.lower()
    return (
        name.endswith(".json")
        or name.endswith(".jsonl")
        or name.endswith(".json.gz")
        or name.endswith(".jsonl.gz")
    )


def open_text(path: Path):
    if path.name.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "r", encoding="utf-8")


def load_json_records_from_file(path: Path) -> list[dict]:
    """
    Supports:
      - one JSON object per .json
      - a list of JSON objects in .json
      - JSONL / JSONL.GZ
    """
    path = Path(path)
    name = path.name.lower()
    records = []

    if name.endswith(".jsonl") or name.endswith(".jsonl.gz"):
        with open_text(path) as f:
            for line_number, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                if not isinstance(obj, Mapping):
                    raise ValueError(f"Expected JSON object at {path}:{line_number}, got {type(obj).__name__}")
                obj = dict(obj)
                obj["_source_path"] = str(path)
                obj["_source_line_number"] = line_number
                records.append(obj)
        return records

    with open_text(path) as f:
        obj = json.load(f)

    if isinstance(obj, Mapping):
        obj = dict(obj)
        obj["_source_path"] = str(path)
        obj["_source_line_number"] = None
        records.append(obj)
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            if not isinstance(item, Mapping):
                raise ValueError(f"Expected JSON object in {path}[{i}], got {type(item).__name__}")
            item = dict(item)
            item["_source_path"] = str(path)
            item["_source_line_number"] = i
            records.append(item)
    else:
        raise ValueError(f"Expected JSON object or list in {path}, got {type(obj).__name__}")

    return records


def load_input_folder(input_dir: Path) -> list[dict]:
    paths = sorted(p for p in Path(input_dir).rglob("*") if p.is_file() and is_json_path(p))
    assert paths, f"No JSON/JSONL files found under {input_dir}"

    all_records = []
    errors = []

    for path in tqdm(paths, desc="load files"):
        try:
            all_records.extend(load_json_records_from_file(path))
        except Exception as e:
            errors.append({"path": str(path), "error": repr(e)})

    if errors:
        display(pd.DataFrame(errors))
        raise RuntimeError(f"{len(errors)} input files failed to load.")

    return all_records


raw_records = load_input_folder(INPUT_DIR)

print("raw records:", len(raw_records))
print("first keys:", list(raw_records[0].keys()) if raw_records else [])

load files:   0%|          | 0/973 [00:00<?, ?it/s]

raw records: 973
first keys: ['file_id', 'entities', 'ocr_boxes', '_source_path', '_source_line_number']


## 5. Generic helpers

In [7]:
def make_with_supported_kwargs(cls, **kwargs):
    """Construct project schema dataclasses/classes while tolerating local field differences."""
    params = inspect.signature(cls).parameters
    usable = {k: v for k, v in kwargs.items() if k in params}
    return cls(**usable)


def short_error(e: Exception) -> str:
    return "".join(format_exception_only(type(e), e)).strip()


def normalize_label(raw_label: str) -> str:
    raw_label = str(raw_label).strip()
    return ENTITY_LABEL_MAP.get(raw_label, raw_label.replace("_", " ").title())


def stable_seed(seed: int, name: str) -> int:
    digest = hashlib.md5(f"{seed}:{name}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16)


def assign_splits(file_ids: list[str]) -> dict[str, str]:
    if SPLIT_SEED is None:
        return {str(file_id): DEFAULT_SPLIT for file_id in file_ids}

    ids = sorted(set(map(str, file_ids)))
    rng = random.Random(stable_seed(SPLIT_SEED, DATASET_NAME))
    rng.shuffle(ids)

    n = len(ids)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    split_map = {}
    for file_id in ids[:n_train]:
        split_map[file_id] = "train"
    for file_id in ids[n_train:n_train + n_val]:
        split_map[file_id] = "val"
    for file_id in ids[n_train + n_val:]:
        split_map[file_id] = "test"

    return split_map


def polygon_to_bbox(points: list[list[float]]) -> list[int]:
    xs = [float(p[0]) for p in points]
    ys = [float(p[1]) for p in points]
    return [
        int(round(min(xs))),
        int(round(min(ys))),
        int(round(max(xs))),
        int(round(max(ys))),
    ]

## 6. OCR reconstruction

In [8]:
def sort_ocr_boxes(ocr_boxes: list[dict]) -> list[dict]:
    enriched = []

    for i, box in enumerate(ocr_boxes or []):
        text = str(box.get("text") or "")
        points = box.get("points")

        if not text or not isinstance(points, list) or len(points) < 4:
            continue

        bbox = polygon_to_bbox(points)
        x0, y0, x1, y1 = bbox

        enriched.append({
            **box,
            "_bbox": bbox,
            "_x0": x0,
            "_y0": y0,
            "_x1": x1,
            "_y1": y1,
            "_source_ocr_index": i,
        })

    # Receipt-style reading order: top-to-bottom, then left-to-right.
    return sorted(enriched, key=lambda b: (b["_y0"], b["_x0"]))


def build_text_and_ocr(sorted_boxes: list[dict]) -> tuple[str, dict]:
    pieces = []
    tokens = []
    lines = []
    paragraphs = []
    cursor = 0

    page_width = 0
    page_height = 0

    for box_index, box in enumerate(sorted_boxes):
        text = str(box.get("text") or "")
        if not text:
            continue

        if pieces:
            pieces.append(JOIN_BOXES_WITH)
            cursor += len(JOIN_BOXES_WITH)

        start = cursor
        pieces.append(text)
        cursor += len(text)
        end = cursor

        x0, y0, x1, y1 = box["_bbox"]
        page_width = max(page_width, x1)
        page_height = max(page_height, y1)

        unit = {
            "text": text,
            "segments": [[start, end]],
            "bbox": [PAGE_ID, x0, y0, x1, y1],
            "page_id": PAGE_ID,
            "orientation": "UP",
            "source_ocr_index": box["_source_ocr_index"],
            "ocr_unit": OCR_UNIT,
        }
        tokens.append(unit)
        lines.append(dict(unit))
        paragraphs.append(dict(unit))

    full_text = "".join(pieces)

    ocr_payload = {
        "text": full_text,
        "pages": [
            {
                "page_id": PAGE_ID,
                "dimension": {
                    "width": int(page_width),
                    "height": int(page_height),
                },
                "tokens": tokens,
                "lines": lines,
                "paragraphs": paragraphs,
            }
        ],
    }

    return full_text, ocr_payload

## 7. Entity span matching

In [9]:
def normalize_with_mapping(text: str) -> tuple[str, list[int]]:
    """
    Collapse whitespace to single spaces while keeping a map from normalized
    character positions back to original character positions.
    """
    normalized_chars = []
    original_positions = []
    in_space = False

    for i, ch in enumerate(str(text)):
        if ch.isspace():
            if not in_space:
                normalized_chars.append(" ")
                original_positions.append(i)
                in_space = True
        else:
            normalized_chars.append(ch)
            original_positions.append(i)
            in_space = False

    start_trim = 0
    end_trim = len(normalized_chars)

    while start_trim < end_trim and normalized_chars[start_trim] == " ":
        start_trim += 1
    while end_trim > start_trim and normalized_chars[end_trim - 1] == " ":
        end_trim -= 1

    normalized = "".join(normalized_chars[start_trim:end_trim])
    mapping = original_positions[start_trim:end_trim]
    return normalized, mapping


def find_entity_span(full_text: str, value: Any) -> dict:
    value = "" if value is None else str(value)

    if not value:
        return {"start": None, "end": None, "text": value, "match_status": "empty_value"}

    # 1) Exact search.
    start = full_text.find(value)
    if start >= 0:
        return {
            "start": start,
            "end": start + len(value),
            "text": full_text[start:start + len(value)],
            "match_status": "exact",
        }

    # 2) Whitespace-normalized search. Needed for multi-box values like address.
    norm_text, mapping = normalize_with_mapping(full_text)
    norm_value, _ = normalize_with_mapping(value)
    norm_start = norm_text.find(norm_value)

    if norm_start >= 0:
        norm_end = norm_start + len(norm_value)
        original_start = mapping[norm_start]
        original_end = mapping[norm_end - 1] + 1
        return {
            "start": original_start,
            "end": original_end,
            "text": full_text[original_start:original_end],
            "match_status": "whitespace_normalized",
        }

    # 3) Numeric substring fallback. Useful for total=14.10 vs OCR="RM 14.10".
    stripped = value.strip()
    if re.fullmatch(r"[-+]?\d+(?:[.,]\d+)?", stripped):
        match = re.search(re.escape(stripped), full_text)
        if match:
            return {
                "start": match.start(),
                "end": match.end(),
                "text": full_text[match.start():match.end()],
                "match_status": "numeric_substring",
            }

    return {"start": None, "end": None, "text": value, "match_status": "not_found"}


def build_labels(full_text: str, entities: Mapping[str, Any]) -> list[dict]:
    labels = []

    for raw_label, value in (entities or {}).items():
        match = find_entity_span(full_text, value)
        label_row = {
            "label": normalize_label(raw_label),
            "start": match["start"],
            "end": match["end"],
            "text": match["text"],
            "raw_label": raw_label,
            "raw_value": "" if value is None else str(value),
            "match_status": match["match_status"],
        }

        if KEEP_UNMATCHED_IN_CANONICAL_LABELS or (label_row["start"] is not None and label_row["end"] is not None):
            labels.append(label_row)

    return labels

## 8. Build canonical `all.jsonl` rows

In [10]:
file_ids = [
    str(r.get("file_id") or r.get("id") or r.get("filename") or i)
    for i, r in enumerate(raw_records)
]
split_map = assign_splits(file_ids)

canonical_rows = []
summary_rows = []
unmatched_rows = []

for i, sample in enumerate(tqdm(raw_records, desc="convert to canonical")):
    file_id = str(sample.get("file_id") or sample.get("id") or sample.get("filename") or i)
    entities = sample.get("entities") or {}
    ocr_boxes = sample.get("ocr_boxes") or []

    sorted_boxes = sort_ocr_boxes(ocr_boxes)
    full_text, ocr_payload = build_text_and_ocr(sorted_boxes)

    # Build all labels once for diagnostics.
    diagnostic_labels = []
    for raw_label, value in (entities or {}).items():
        match = find_entity_span(full_text, value)
        diagnostic_labels.append({
            "label": normalize_label(raw_label),
            "start": match["start"],
            "end": match["end"],
            "text": match["text"],
            "raw_label": raw_label,
            "raw_value": "" if value is None else str(value),
            "match_status": match["match_status"],
        })

    labels = [
        x for x in diagnostic_labels
        if KEEP_UNMATCHED_IN_CANONICAL_LABELS or (x["start"] is not None and x["end"] is not None)
    ]

    n_unmatched = sum(1 for x in diagnostic_labels if x["start"] is None or x["end"] is None)
    n_matched = len(diagnostic_labels) - n_unmatched

    for lab in diagnostic_labels:
        if lab["start"] is None or lab["end"] is None:
            unmatched_rows.append({
                "file_id": file_id,
                "label": lab["label"],
                "raw_label": lab["raw_label"],
                "raw_value": lab["raw_value"],
                "match_status": lab["match_status"],
                "source_path": sample.get("_source_path"),
                "source_line_number": sample.get("_source_line_number"),
            })

    row = {
        "original_filename": file_id,
        "ocr": None,
        "text": full_text,
        "labels": json.dumps(labels, ensure_ascii=False),
        "image_files": json.dumps([], ensure_ascii=False),
        "OCR": ocr_payload,
        "dataset": DATASET_NAME,
        "split": split_map[file_id],
        "n_annotations": sum(1 for x in labels if x["start"] is not None and x["end"] is not None),
        "n_unmatched_annotations": n_unmatched,
        "n_ocr_boxes": len(ocr_boxes),
        "n_ocr_tokens": len(ocr_payload["pages"][0]["tokens"]),
        "ocr_unit": OCR_UNIT,
        "source_path": sample.get("_source_path"),
        "source_line_number": sample.get("_source_line_number"),
    }

    canonical_rows.append(row)

    summary_rows.append({
        "file_id": file_id,
        "status": "ok",
        "n_ocr_boxes": len(ocr_boxes),
        "n_tokens": len(ocr_payload["pages"][0]["tokens"]),
        "text_len": len(full_text),
        "n_entities": len(entities),
        "n_labels_written": len(labels),
        "n_matched": n_matched,
        "n_unmatched": n_unmatched,
        "source_path": sample.get("_source_path"),
    })

canonical_df = pd.DataFrame(canonical_rows)
summary_df = pd.DataFrame(summary_rows)
unmatched_df = pd.DataFrame(unmatched_rows)

print("canonical rows:", len(canonical_df))
print("raw records:", len(raw_records))
print("total unmatched entities:", len(unmatched_df))

display(canonical_df.head())
display(summary_df.head())
display(unmatched_df.head(50))

assert not canonical_df.empty, "No canonical rows created."
assert int((canonical_df["n_ocr_tokens"] == 0).sum()) == 0, "Some records have zero OCR tokens."

convert to canonical:   0%|          | 0/973 [00:00<?, ?it/s]

canonical rows: 973
raw records: 973
total unmatched entities: 262


,original_filename,ocr,text,labels,image_files,OCR,dataset,split,n_annotations,n_unmatched_annotations,n_ocr_boxes,n_ocr_tokens,ocr_unit,source_path,source_line_number
0,X00016469612,None,TAN WOON YANN\nBOOK TA .K(TAMAN DAYA) SDN BND\...,"[{""label"": ""Date"", ""start"": 156, ""end"": 166, ""...",[],{'text': 'TAN WOON YANN BOOK TA .K(TAMAN DAYA)...,invoices,train,3,1,44,44,box_text,/home/vios/PycharmProjects/serialization-strat...,None
1,X00016469619,None,"TAN WOON YANN\nINDAH GIFT & HOME DECO\n27,JALA...","[{""label"": ""Company"", ""start"": 14, ""end"": 36, ...",[],{'text': 'TAN WOON YANN INDAH GIFT & HOME DECO...,invoices,train,3,1,48,48,box_text,/home/vios/PycharmProjects/serialization-strat...,None
2,X00016469620,None,TAN WOON YANN\nMR D.T.Y. (JOHOR) SDN BHD\n(CO....,"[{""label"": ""Date"", ""start"": 600, ""end"": 608, ""...",[],{'text': 'TAN WOON YANN MR D.T.Y. (JOHOR) SDN ...,invoices,train,3,1,54,54,box_text,/home/vios/PycharmProjects/serialization-strat...,None
3,X00016469622,None,TAN WOON YANN\nYONGFATT ENTERPRISE\n(JM0517726...,"[{""label"": ""Company"", ""start"": 14, ""end"": 33, ...",[],{'text': 'TAN WOON YANN YONGFATT ENTERPRISE (J...,invoices,train,3,1,60,60,box_text,/home/vios/PycharmProjects/serialization-strat...,None
4,X00016469623,None,TAN WOON YANN\nMR D.I.Y. (M) SDN BHD\n(CO. RFG...,"[{""label"": ""Company"", ""start"": 14, ""end"": 35, ...",[],{'text': 'TAN WOON YANN MR D.I.Y. (M) SDN BHD ...,invoices,train,4,0,61,61,box_text,/home/vios/PycharmProjects/serialization-strat...,None


,file_id,status,n_ocr_boxes,n_tokens,text_len,n_entities,n_labels_written,n_matched,n_unmatched,source_path
0,X00016469612,ok,44,44,485,4,3,3,1,/home/vios/PycharmProjects/serialization-strat...
1,X00016469619,ok,48,48,684,4,3,3,1,/home/vios/PycharmProjects/serialization-strat...
2,X00016469620,ok,54,54,723,4,3,3,1,/home/vios/PycharmProjects/serialization-strat...
3,X00016469622,ok,60,60,584,4,3,3,1,/home/vios/PycharmProjects/serialization-strat...
4,X00016469623,ok,61,61,797,4,4,4,0,/home/vios/PycharmProjects/serialization-strat...


,file_id,label,raw_label,raw_value,match_status,source_path,source_line_number
0,X00016469612,Company,company,BOOK TA .K (TAMAN DAYA) SDN BHD,not_found,/home/vios/PycharmProjects/serialization-strat...,None
1,X00016469619,Address,address,"27, JALAN DEDAP 13, TAMAN JOHOR JAYA, 81100 JO...",not_found,/home/vios/PycharmProjects/serialization-strat...,None
2,X00016469620,Company,company,MR D.I.Y. (JOHOR) SDN BHD,not_found,/home/vios/PycharmProjects/serialization-strat...,None
3,X00016469622,Address,address,NO 122.124. JALAN DEDAP 13 81100 JOHOR BAHRU,not_found,/home/vios/PycharmProjects/serialization-strat...,None
4,X00016469669,Address,address,"NO.2&4, JALAN HARMONI 3/2, TAMAN DESA HARMONI....",not_found,/home/vios/PycharmProjects/serialization-strat...,None
5,X00016469670,Address,address,"NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, B175...",not_found,/home/vios/PycharmProjects/serialization-strat...,None
6,X00016469672,Address,address,"NO.53 JALAN PUTRA 1, TAMAN SRI PUTRA, 81200 JO...",not_found,/home/vios/PycharmProjects/serialization-strat...,None
7,X51005230605,Address,address,"KM 458.4 BKT LANJAN UTARA, L/RAYA UTARA SELATA...",not_found,/home/vios/PycharmProjects/serialization-strat...,None
8,X51005230621,Address,address,"LOT 13, JALAN IPOH, KG BATU 30, ULU YAM LAMA 4...",not_found,/home/vios/PycharmProjects/serialization-strat...,None
9,X51005230648,Address,address,"47, JALAN MERANTI 1, SEK. 3, BANDAR UTAMA BATA...",not_found,/home/vios/PycharmProjects/serialization-strat...,None


## 9. Write canonical files

## 10. Convert canonical rows to `CanonicalDocument`

In [11]:
def parse_labels(value) -> list[dict]:
    if isinstance(value, str):
        return json.loads(value)
    if isinstance(value, list):
        return value
    return []


def ocr_page_size(ocr: Mapping[str, Any], page_id: int = 0) -> tuple[int | None, int | None]:
    pages = ocr.get("pages") or []
    if not pages:
        return None, None
    page = pages[page_id] if page_id < len(pages) else pages[0]
    dim = page.get("dimension") or {}
    return dim.get("width"), dim.get("height")


def bbox_without_page(bbox):
    vals = list(bbox)
    if len(vals) == 5:
        vals = vals[1:]
    return [int(round(float(v))) for v in vals[:4]]


def tokens_from_ocr_payload(ocr: Mapping[str, Any]) -> list[OCRToken]:
    out = []
    pages = ocr.get("pages") or []

    for page_index, page in enumerate(pages):
        page_id = int(page.get("page_id", page_index))
        dim = page.get("dimension") or {}
        page_width = dim.get("width")
        page_height = dim.get("height")

        for raw in page.get("tokens", []) or []:
            text = str(raw.get("text") or "")
            segments = raw.get("segments") or []
            if not text or not segments:
                continue
            start, end = int(segments[0][0]), int(segments[0][1])
            bbox = bbox_without_page(raw.get("bbox"))

            out.append(
                make_with_supported_kwargs(
                    OCRToken,
                    text=text,
                    start=start,
                    end=end,
                    page=page_id,
                    bbox=bbox,
                    page_width=page_width,
                    page_height=page_height,
                    style={},
                    extra={
                        "ocr_unit": raw.get("ocr_unit", OCR_UNIT),
                        "source_ocr_index": raw.get("source_ocr_index"),
                    },
                )
            )

    return sorted(out, key=lambda t: (getattr(t, "page", 0), getattr(t, "start", 0), getattr(t, "end", 0)))


def blocks_from_ocr_payload(ocr: Mapping[str, Any]) -> list[OCRBlock]:
    out = []
    pages = ocr.get("pages") or []

    for page_index, page in enumerate(pages):
        page_id = int(page.get("page_id", page_index))
        dim = page.get("dimension") or {}
        page_width = dim.get("width")
        page_height = dim.get("height")
        block_id = 0

        # The converter stores box-level units in lines/paragraphs.
        for key in ["lines", "paragraphs"]:
            for raw in page.get(key, []) or []:
                text = str(raw.get("text") or "")
                segments = raw.get("segments") or []
                if not text or not segments:
                    continue
                start, end = int(segments[0][0]), int(segments[0][1])
                bbox = bbox_without_page(raw.get("bbox"))

                out.append(
                    make_with_supported_kwargs(
                        OCRBlock,
                        text=text,
                        start=start,
                        end=end,
                        page=page_id,
                        bbox=bbox,
                        block_id=block_id,
                        block_type=key[:-1],
                        page_width=page_width,
                        page_height=page_height,
                        extra={"ocr_unit": raw.get("ocr_unit", OCR_UNIT)},
                    )
                )
                block_id += 1

    return sorted(out, key=lambda b: (getattr(b, "page", 0), getattr(b, "start", 0), getattr(b, "end", 0)))


def row_to_document(row: pd.Series) -> CanonicalDocument:
    labels = parse_labels(row["labels"])
    text = row["text"]
    ocr = row["OCR"]

    annotations = []
    for lab in labels:
        if lab.get("start") is None or lab.get("end") is None:
            continue
        start, end = int(lab["start"]), int(lab["end"])
        annotations.append(
            make_with_supported_kwargs(
                Annotation,
                label=str(lab["label"]),
                raw_label=str(lab.get("raw_label", lab["label"])),
                start=start,
                end=end,
                text=str(lab.get("text") or text[start:end]),
                extra={
                    "raw_value": lab.get("raw_value"),
                    "match_status": lab.get("match_status"),
                },
            )
        )

    return make_with_supported_kwargs(
        CanonicalDocument,
        doc_id=str(row["original_filename"]),
        dataset_name=str(row["dataset"]),
        text=text,
        tokens=tokens_from_ocr_payload(ocr),
        blocks=blocks_from_ocr_payload(ocr),
        annotations=annotations,
        metadata={
            "split": row["split"],
            "ocr_unit": row.get("ocr_unit", OCR_UNIT),
            "source_path": row.get("source_path"),
        },
    )


docs = []
doc_errors = []

for _, row in tqdm(canonical_df.iterrows(), total=len(canonical_df), desc="CanonicalDocument"):
    try:
        docs.append(row_to_document(row))
    except Exception as e:
        doc_errors.append({
            "doc_id": row.get("original_filename"),
            "error": short_error(e),
        })

doc_diag_df = pd.DataFrame([
    {
        "doc_id": d.doc_id,
        "split": d.metadata.get("split"),
        "n_tokens": len(d.tokens),
        "n_blocks": len(d.blocks or []),
        "n_annotations": len(d.annotations or []),
        "text_len": len(d.text or ""),
    }
    for d in docs
])

doc_errors_df = pd.DataFrame(doc_errors)

display(doc_diag_df.head())
display(doc_errors_df.head(50))

assert len(docs) > 0, "No CanonicalDocument objects created."
assert int((doc_diag_df["n_tokens"] == 0).sum()) == 0, "Some docs have zero tokens."
assert int((doc_diag_df["n_annotations"] == 0).sum()) < len(doc_diag_df), "All docs have zero annotations."

CanonicalDocument:   0%|          | 0/973 [00:00<?, ?it/s]

,doc_id,split,n_tokens,n_blocks,n_annotations,text_len
0,X00016469612,train,44,88,3,485
1,X00016469619,train,48,96,3,684
2,X00016469620,train,54,108,3,723
3,X00016469622,train,60,120,3,584
4,X00016469623,train,61,122,4,797


""


## 11. Alignment and BIO labels

In [12]:
prepared_docs = []
alignment_rows = []

for i, doc in enumerate(tqdm(docs, desc="align BIO labels")):
    try:
        quality = None
        if i < ASSESS_QUALITY_FIRST_N_DOCS:
            quality = assess_document_quality(doc, min_token_overlap=MIN_TOKEN_OVERLAP)

        filtered = filter_annotations_for_training(
            doc,
            min_token_overlap=MIN_TOKEN_OVERLAP,
            drop_unavailable=DROP_UNAVAILABLE,
            drop_loose_mismatch=DROP_LOOSE_MISMATCH,
            attach_quality_report=quality is not None,
        )

        if not filtered.annotations:
            raise ValueError("No annotations left after filtering.")

        labeled = add_token_labels(
            filtered,
            min_token_overlap=MIN_TOKEN_OVERLAP,
            conflict_policy="keep_first",
            include_alignment_diagnostics=False,
        )

        token_labels = labeled.metadata["token_labels"]
        prepared_docs.append(labeled)

        alignment_rows.append({
            "doc_id": doc.doc_id,
            "split": doc.metadata.get("split"),
            "status": "ok",
            "n_tokens": len(labeled.tokens),
            "n_annotations_before": len(doc.annotations or []),
            "n_annotations_after": len(labeled.annotations or []),
            "n_non_o_token_labels": sum(1 for x in token_labels if x != "O"),
            "token_available_rate": None if quality is None else quality.token_available_rate,
            "strict_match_rate": None if quality is None else quality.strict_match_rate,
            "loose_value_match_rate": None if quality is None else quality.loose_value_match_rate,
            "error": None,
        })

    except Exception as e:
        alignment_rows.append({
            "doc_id": doc.doc_id,
            "split": doc.metadata.get("split"),
            "status": "error",
            "n_tokens": len(doc.tokens),
            "n_annotations_before": len(doc.annotations or []),
            "n_annotations_after": None,
            "n_non_o_token_labels": None,
            "token_available_rate": None,
            "strict_match_rate": None,
            "loose_value_match_rate": None,
            "error": short_error(e),
        })

alignment_df = pd.DataFrame(alignment_rows)

display(alignment_df.head())
display(
    alignment_df.groupby("status")
    .agg(
        n_docs=("doc_id", "count"),
        mean_tokens=("n_tokens", "mean"),
        mean_annotations_after=("n_annotations_after", "mean"),
        mean_non_o=("n_non_o_token_labels", "mean"),
        mean_token_available_rate=("token_available_rate", "mean"),
        mean_loose_value_match_rate=("loose_value_match_rate", "mean"),
    )
    .reset_index()
)
display(alignment_df[alignment_df["status"] == "error"].head(100))

assert len(prepared_docs) > 0, "No documents survived BIO alignment."

align BIO labels:   0%|          | 0/973 [00:00<?, ?it/s]

,doc_id,split,status,n_tokens,n_annotations_before,n_annotations_after,n_non_o_token_labels,token_available_rate,strict_match_rate,loose_value_match_rate,error
0,X00016469612,train,ok,44,3,2,5,0.666667,0.666667,0.666667,None
1,X00016469619,train,ok,48,3,2,2,0.666667,0.666667,0.666667,None
2,X00016469620,train,ok,54,3,2,5,0.666667,0.333333,0.666667,None
3,X00016469622,train,ok,60,3,3,3,1.000000,1.000000,1.000000,None
4,X00016469623,train,ok,61,4,3,6,0.750000,0.500000,0.750000,None


,status,n_docs,mean_tokens,mean_annotations_after,mean_non_o,mean_token_available_rate,mean_loose_value_match_rate
0,ok,973,53.783145,3.295992,4.628983,0.823333,0.823333


,doc_id,split,status,n_tokens,n_annotations_before,n_annotations_after,n_non_o_token_labels,token_available_rate,strict_match_rate,loose_value_match_rate,error


## 12. Instantiate strategy converters / serializers

In [13]:
all_serializers = {
    "plain_text": PlainTextSerializer(),
    "page_aware": PageAwareSerializer(),
    "block_aware": BlockAwareSerializer(),
    "line_aware": LineAwareSerializer(),
    "precedence_graph_order": PrecedenceGraphOrderSerializer(),
    "key_value_row_pairs": KeyValueRowPairsSerializer(),
    "key_value_anchor_pairs": KeyValueAnchorPairsSerializer(),
    "column_aware": ColumnAwareSerializer(
        min_gap_ratio=0.035,
        min_tokens_per_column=8,
        max_columns=4,
    ),
    "xycut_aware": XYCutAwareSerializer(
        min_gap_ratio_x=0.06,
        min_gap_ratio_y=0.035,
        min_tokens_per_region=8,
        max_depth=6,
    ),
    "lmdx_coord_suffix": LMDXCoordSuffixSerializer(
        n_buckets=100,
        coord_mode="center",
    ),
    "compact_bbox_token": CompactBBoxTokenSerializer(
        n_buckets=100,
        position="prefix",
    ),

    # Heavy ablations.
    "rowcol_bucket": RowColBucketSerializer(n_buckets=100),
    "bbox_token": BBoxTokenSerializer(n_buckets=100),
}

serializers = {
    name: serializer
    for name, serializer in all_serializers.items()
    if INCLUDE_HEAVY_SERIALIZERS or name not in HEAVY_SERIALIZERS
}

print("selected serializers:", sorted(serializers))

selected serializers: ['bbox_token', 'block_aware', 'column_aware', 'compact_bbox_token', 'line_aware', 'lmdx_coord_suffix', 'page_aware', 'plain_text', 'rowcol_bucket', 'xycut_aware']


## 13. Serialization helpers

In [14]:
def validate_serialized_record(record: dict, doc: CanonicalDocument, token_labels: list[str]) -> None:
    n = len(record["tokens"])
    same_length_keys = [
        "source_token_indices",
        "loss_mask",
        "layout_roles",
        "item_attrs",
        "labels",
        "pages",
        "bboxes",
        "normalized_bboxes",
        "offsets",
    ]

    for key in same_length_keys:
        assert key in record, f"missing {key}"
        assert len(record[key]) == n, f"{record['serializer']} {key} length mismatch"

    real_indices = [idx for idx in record["source_token_indices"] if idx is not None]
    assert sorted(real_indices) == list(range(len(doc.tokens))), "source_token_indices do not cover original tokens exactly once"

    for pos, idx in enumerate(record["source_token_indices"]):
        if idx is None:
            assert record["loss_mask"][pos] is False
            assert record["labels"][pos] == IGNORE_LABEL
        else:
            assert record["loss_mask"][pos] is True
            assert record["labels"][pos] == token_labels[idx]

    collapsed = collapse_serialized_labels_to_original(record, record["labels"])
    assert collapsed == token_labels, "BIO round-trip failed"


def add_serialized_metadata(record: dict, doc: CanonicalDocument) -> dict:
    record = dict(record)
    record["split"] = doc.metadata.get("split")
    record["doc_id"] = doc.doc_id
    record["split_seed"] = SPLIT_SEED
    record["ocr_unit"] = doc.metadata.get("ocr_unit", OCR_UNIT)

    record.setdefault("metadata", {})
    if isinstance(record["metadata"], dict):
        record["metadata"].update({
            "split": doc.metadata.get("split"),
            "split_seed": SPLIT_SEED,
            "ocr_unit": doc.metadata.get("ocr_unit", OCR_UNIT),
            "source_path": doc.metadata.get("source_path"),
        })

    return record


class LazyJsonlWriter:
    def __init__(self, root: Path):
        self.root = Path(root)
        self.handles = {}
        self.counts = Counter()

    def path_for(self, serializer: str, split: str) -> Path:
        return self.root / "serialized" / serializer / f"{split}.jsonl"

    def write(self, serializer: str, split: str, record: dict) -> None:
        key = (serializer, split)
        if key not in self.handles:
            path = self.path_for(serializer, split)
            path.parent.mkdir(parents=True, exist_ok=True)
            self.handles[key] = open(path, "w", encoding="utf-8")

        self.handles[key].write(json.dumps(record, ensure_ascii=False) + "\n")
        self.counts[key] += 1

    def close(self):
        for handle in self.handles.values():
            handle.flush()
            handle.close()
        self.handles = {}

    def summary(self) -> pd.DataFrame:
        rows = []
        for (serializer, split), n in sorted(self.counts.items()):
            path = self.path_for(serializer, split)
            rows.append({
                "serializer": serializer,
                "split": split,
                "n_records": n,
                "path": str(path),
                "exists": path.exists(),
                "size_bytes": path.stat().st_size if path.exists() else 0,
            })
        return pd.DataFrame(rows)

## 14. Apply serializers and write evaluation-ready JSONL

In [15]:
writer = LazyJsonlWriter(OUTPUT_ROOT) if WRITE_SERIALIZED else None

stats_rows = []
error_rows = []
preview_records = {}

try:
    for doc_index, doc in enumerate(tqdm(prepared_docs, desc="serialize strategies")):
        token_labels = doc.metadata["token_labels"]
        validate_this = doc_index < VALIDATE_FIRST_N_DOCS_PER_DATASET
        split = doc.metadata.get("split", DEFAULT_SPLIT)

        for serializer_name, serializer in serializers.items():
            try:
                t0 = time.perf_counter()
                record = serializer.serialize_train(doc)
                seconds = time.perf_counter() - t0

                if validate_this:
                    validate_serialized_record(record, doc, token_labels)

                record = add_serialized_metadata(record, doc)

                if writer is not None:
                    writer.write(serializer_name, "all", record)
                    if split != "all":
                        writer.write(serializer_name, split, record)

                preview_records.setdefault(serializer_name, record)

                real = sum(record["loss_mask"])
                stats_rows.append({
                    "dataset": doc.dataset_name,
                    "doc_id": doc.doc_id,
                    "split": split,
                    "serializer": serializer_name,
                    "validated": validate_this,
                    "original_token_count": len(doc.tokens),
                    "serialized_token_count": len(record["tokens"]),
                    "real_token_count": real,
                    "layout_token_count": len(record["tokens"]) - real,
                    "length_multiplier": len(record["tokens"]) / max(1, len(doc.tokens)),
                    "n_annotations": len(doc.annotations or []),
                    "n_non_o_token_labels": sum(1 for x in token_labels if x != "O"),
                    "serialize_seconds": seconds,
                })

            except Exception as e:
                error_rows.append({
                    "doc_id": doc.doc_id,
                    "serializer": serializer_name,
                    "error": short_error(e),
                })

finally:
    if writer is not None:
        writer.close()

stats_df = pd.DataFrame(stats_rows)
errors_df = pd.DataFrame(error_rows)

display(stats_df.head())
display(errors_df.head(50))

print("serialized rows:", len(stats_df))
print("serialization errors:", len(errors_df))

assert not stats_df.empty, "No serialized records created."

serialize strategies:   0%|          | 0/973 [00:00<?, ?it/s]

,dataset,doc_id,split,serializer,validated,original_token_count,serialized_token_count,real_token_count,layout_token_count,length_multiplier,n_annotations,n_non_o_token_labels,serialize_seconds
0,invoices,X00016469612,train,plain_text,True,44,44,44,0,1.000000,2,5,0.000375
1,invoices,X00016469612,train,page_aware,True,44,45,44,1,1.022727,2,5,0.000250
2,invoices,X00016469612,train,block_aware,True,44,89,44,45,2.022727,2,5,0.001658
3,invoices,X00016469612,train,line_aware,True,44,72,44,28,1.636364,2,5,0.000629
4,invoices,X00016469612,train,column_aware,True,44,46,44,2,1.045455,2,5,0.000640


""


serialized rows: 9730
serialization errors: 0


## 15. Reports and diagnostics

## 16. Inspect one serialized record

In [16]:
if not preview_records:
    print("No preview records.")
else:
    serializer_name = "xycut_aware" if "xycut_aware" in preview_records else next(iter(preview_records))
    record = preview_records[serializer_name]

    print("serializer:", record["serializer"])
    print("doc_id:", record["doc_id"])
    print("split:", record["split"])
    print("tokens:", len(record["tokens"]))

    inspect_df = pd.DataFrame({
        "pos": list(range(len(record["tokens"]))),
        "token": record["tokens"],
        "label": record["labels"],
        "loss_mask": record["loss_mask"],
        "source_token_index": record["source_token_indices"],
        "layout_role": record["layout_roles"],
        "page": record["pages"],
        "bbox": record["bboxes"],
    })

    display(inspect_df.head(200))

serializer: xycut_aware
doc_id: X00016469612
split: train
tokens: 46


,pos,token,label,loss_mask,source_token_index,layout_role,page,bbox
0,0,[PAGE_0],-100,False,NaN,page,0,None
1,1,[REGION_0],-100,False,NaN,xycut_region,0,None
2,2,TAN WOON YANN,O,True,0.0,ocr_token,0,"[72, 25, 326, 64]"
3,3,BOOK TA .K(TAMAN DAYA) SDN BND,O,True,1.0,ocr_token,0,"[50, 82, 440, 121]"
4,4,789417-W,O,True,2.0,ocr_token,0,"[205, 121, 285, 139]"
5,5,"NO.53 55,57 & 59, JALAN SAGU 18,",B-address,True,3.0,ocr_token,0,"[110, 144, 383, 163]"
6,6,"TAMAN DAYA,",I-address,True,4.0,ocr_token,0,"[192, 169, 299, 187]"
7,7,"81100 JOHOR BAHRU,",I-address,True,5.0,ocr_token,0,"[162, 193, 334, 211]"
8,8,JOHOR.,I-address,True,6.0,ocr_token,0,"[217, 216, 275, 233]"
9,9,DOCUMENT NO : TD01167104,O,True,7.0,ocr_token,0,"[50, 342, 279, 359]"


## 17. Final checks

In [17]:
checks = {
    "raw_loaded": len(raw_records) > 0,
    "canonical_rows_created": not canonical_df.empty,
    "documents_created": len(docs) > 0,
    "prepared_docs_created": len(prepared_docs) > 0,
    "serialized_records_created": not stats_df.empty,
    "no_doc_errors": doc_errors_df.empty,
    "no_serialization_errors": errors_df.empty,
}

checks["functional_pass"] = (
    checks["raw_loaded"]
    and checks["canonical_rows_created"]
    and checks["documents_created"]
    and checks["prepared_docs_created"]
    and checks["serialized_records_created"]
)

print(json.dumps(checks, indent=2))

print("Serialized root:", OUTPUT_ROOT / "serialized")

if checks["functional_pass"]:
    print("\nFUNCTIONAL PASS: dataset is ready for comparative evaluation.")
else:
    print("\nFAIL/WARN: inspect diagnostics CSVs.")

{
  "raw_loaded": true,
  "canonical_rows_created": true,
  "documents_created": true,
  "prepared_docs_created": true,
  "serialized_records_created": true,
  "no_doc_errors": true,
  "no_serialization_errors": true,
  "functional_pass": true
}
Serialized root: /home/vios/PycharmProjects/serialization-strategies/data/processed/multi_docs/serialized

FUNCTIONAL PASS: dataset is ready for comparative evaluation.
